# Object-Level Paired Test: Does L Carry Unique Information vs R?

Pairs at the **object** level, not the image level. Since RGB and LGB
share identical images and identical label files, every GT box has one
score from the RGB model and one from the LGB model, for the same
physical object -- ~4628 pairs instead of 500.

Two comparisons, testing two different claims:

1. **Only R vs Only L** -- is a single channel (R alone / L alone)
   informative on its own? Tests "L captures more detection-relevant
   signal than R does, in isolation."
2. **Mask R vs Mask L** -- with G and B still present, does removing R
   hurt less than removing L? If yes, that means G/B can substitute for
   R but not for L -- the sharper claim: **L carries information G/B
   don't already have**, i.e., genuinely unique/non-redundant information.


In [9]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import csv
from scipy import stats
from ultralytics import YOLO


In [10]:
DATASET = 'SIMD'
ARCH_LABEL = 'YOLOv8n'

base_path = r"C:\Users\DELL\Desktop\c"

pt_paths = {
    'RGB': os.path.join(base_path, rf'runs\detect\runs\{DATASET}_RGB_{ARCH_LABEL}_weighted\weights\best.pt'),
    'LGB': os.path.join(base_path, rf'runs\detect\runs\{DATASET}_LGB_{ARCH_LABEL}_weighted\weights\best.pt'),
}
val_img_paths = {
    'RGB': os.path.join(base_path, rf"{DATASET}_RGB\test\images"),
    'LGB': os.path.join(base_path, rf"{DATASET}_LGB\test\images"),
}

NUM_CLASSES = 15
CLASS_NAMES = {
    0: 'car', 1: 'truck', 2: 'van', 3: 'longvehicle', 4: 'bus',
    5: 'airliner', 6: 'propeller', 7: 'trainer', 8: 'chartered',
    9: 'fighter', 10: 'other', 11: 'stairtruck', 12: 'pushbacktruck',
    13: 'helicopter', 14: 'boat',
}

PREDICT_CONF = 0.001
IOU_THR = 0.5
FILL_MODE = 'zero'

# Channel 3 (array index 2, BGR layout) = R for RGB model, L for LGB model
CONFIGS = {
    'Only': {'channel': 3, 'type': 'keep_only'},   # Only R  vs  Only L
    'Mask': {'channel': 3, 'type': 'zero_ch'},     # Mask R  vs  Mask L
}


In [11]:
def apply_channel_mask(img, channel, mask_type, fill_mode='zero'):
    out = img.copy()
    def fill(c):
        out[:, :, c] = 0 if fill_mode == 'zero' else int(round(img[:, :, c].mean()))
    if mask_type == 'zero_ch':
        fill(channel - 1)
    elif mask_type == 'keep_only':
        for c in range(3):
            if c != channel - 1:
                fill(c)
    return out


def label_path_for_image(img_path):
    base, _ = os.path.splitext(img_path)
    base = base.replace(os.sep + 'images' + os.sep, os.sep + 'labels' + os.sep)
    return base + '.txt'


def get_gt_boxes(lbl_path, img_w, img_h, num_classes):
    boxes = []
    if not os.path.exists(lbl_path):
        return boxes
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            if not (0 <= c < num_classes):
                continue
            xc, yc, w, h = map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((c, (x1, y1, x2, y2)))
    return boxes


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def match_greedy(gt_boxes, preds, iou_thr):
    """Returns one score per GT box, in the SAME order as gt_boxes.
    This order-preservation is what makes object-level pairing valid: as
    long as both models are evaluated against the same gt_boxes list
    (same label file, same order), position i in RGB's result list and
    position i in LGB's result list are guaranteed to be the same object.
    """
    preds_sorted = sorted(preds, key=lambda x: -x[1])
    n = len(gt_boxes)
    scores = [0.0] * n
    claimed = set()
    for pc, conf, pbox in preds_sorted:
        best_iou, best_i = 0.0, -1
        for i, (gc, gbox) in enumerate(gt_boxes):
            if i in claimed or gc != pc:
                continue
            v = iou(gbox, pbox)
            if v >= iou_thr and v > best_iou:
                best_iou, best_i = v, i
        if best_i >= 0:
            claimed.add(best_i)
            scores[best_i] = conf
    return scores


## Build the shared image list (RGB and LGB share identical filenames)


In [12]:
rgb_files = set(f for f in os.listdir(val_img_paths['RGB']) if f.endswith(('.jpg', '.jpeg', '.png')))
lgb_files = set(f for f in os.listdir(val_img_paths['LGB']) if f.endswith(('.jpg', '.jpeg', '.png')))
assert rgb_files == lgb_files, "RGB and LGB folders don't share identical filenames -- object-level pairing is invalid."

img_list = sorted(f for f in rgb_files if os.path.exists(label_path_for_image(os.path.join(val_img_paths['RGB'], f))))
print(f"{len(img_list)} shared images")


500 shared images


## Main loop

For each image: load GT boxes ONCE (identical for both models, same
label file). For each config ('Only', 'Mask'), run RGB masked on channel
3 and LGB masked on channel 3, and record every object's pair of scores
explicitly keyed by (image, box_index, class) -- explicit keys, not
just relying on list order, so a skipped/corrupt image in one folder
can't silently misalign the pairing.


In [13]:
yolo_models = {'RGB': YOLO(pt_paths['RGB']), 'LGB': YOLO(pt_paths['LGB'])}

# paired_records[config_name] = list of dicts: {image, box_idx, class, score_R, score_L}
paired_records = {name: [] for name in CONFIGS}

for img_name in tqdm(img_list, desc="Processing"):
    rgb_path = os.path.join(val_img_paths['RGB'], img_name)
    lgb_path = os.path.join(val_img_paths['LGB'], img_name)

    img_rgb = cv2.imread(rgb_path)
    img_lgb = cv2.imread(lgb_path)
    if img_rgb is None or img_lgb is None:
        continue  # skip this image for BOTH models -- keeps pairing intact

    H, W = img_rgb.shape[:2]
    gt_boxes = get_gt_boxes(label_path_for_image(rgb_path), W, H, NUM_CLASSES)
    if not gt_boxes:
        continue

    for cfg_name, cfg in CONFIGS.items():
        img_r = apply_channel_mask(img_rgb, cfg['channel'], cfg['type'], FILL_MODE)
        img_l = apply_channel_mask(img_lgb, cfg['channel'], cfg['type'], FILL_MODE)

        preds_r, preds_l = [], []
        for model_name, img_in, preds_out in [('RGB', img_r, preds_r), ('LGB', img_l, preds_l)]:
            res = yolo_models[model_name].predict(img_in, imgsz=640, conf=PREDICT_CONF, verbose=False)
            boxes = res[0].boxes
            if boxes is not None and len(boxes) > 0:
                for c, conf, bb in zip(boxes.cls.cpu().numpy(), boxes.conf.cpu().numpy(), boxes.xyxy.cpu().numpy()):
                    preds_out.append((int(c), float(conf), tuple(map(float, bb))))

        scores_r = match_greedy(gt_boxes, preds_r, IOU_THR)
        scores_l = match_greedy(gt_boxes, preds_l, IOU_THR)

        for i, (gt_c, _) in enumerate(gt_boxes):
            paired_records[cfg_name].append({
                'image': img_name, 'box_idx': i, 'class': gt_c,
                'score_R': scores_r[i], 'score_L': scores_l[i],
            })


Processing: 100%|████████████████████████████████████████████████████████████████████| 500/500 [00:27<00:00, 18.15it/s]


## Paired significance tests

Run separately for 'Only' (Only R vs Only L) and 'Mask' (Mask R vs Mask
L), both pooled across all classes and broken out per class.


In [14]:
def run_tests(records, label):
    r_scores = np.array([rec['score_R'] for rec in records])
    l_scores = np.array([rec['score_L'] for rec in records])
    diffs = r_scores - l_scores

    t_stat, p_t = stats.ttest_rel(r_scores, l_scores)
    try:
        w_stat, p_w = stats.wilcoxon(r_scores, l_scores)
    except ValueError:
        w_stat, p_w = float('nan'), float('nan')
    cohens_d = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else float('nan')

    print(f"\n{label}  (N={len(records)} objects)")
    print(f"  mean R={r_scores.mean():.4f}  mean L={l_scores.mean():.4f}  diff(R-L)={diffs.mean():+.4f}")
    print(f"  paired t-test:  t={t_stat:.4f}, p={p_t:.6f} {'(SIGNIFICANT)' if p_t < 0.05 else '(not significant)'}")
    print(f"  Wilcoxon:       W={w_stat:.4f}, p={p_w:.6f} {'(SIGNIFICANT)' if p_w < 0.05 else '(not significant)'}")
    print(f"  Cohen's d:      {cohens_d:.4f}")
    return [label, len(records), r_scores.mean(), l_scores.mean(), diffs.mean(),
            t_stat, p_t, w_stat, p_w, cohens_d]

summary = []
summary.append(run_tests(paired_records['Only'], 'Only R vs Only L (pooled, all classes)'))
summary.append(run_tests(paired_records['Mask'], 'Mask R vs Mask L (pooled, all classes)'))

print("\n--- Per-class breakdown ---")
for cfg_name in CONFIGS:
    print(f"\n=== {cfg_name} ===")
    for c in range(NUM_CLASSES):
        recs = [r for r in paired_records[cfg_name] if r['class'] == c]
        if len(recs) < 5:
            continue  # too few pairs for a meaningful test
        row = run_tests(recs, f"{cfg_name}: {CLASS_NAMES[c]}")
        summary.append(row)



Only R vs Only L (pooled, all classes)  (N=4628 objects)
  mean R=0.4598  mean L=0.3428  diff(R-L)=+0.1170
  paired t-test:  t=30.5152, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=2050977.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      0.4486

Mask R vs Mask L (pooled, all classes)  (N=4628 objects)
  mean R=0.5873  mean L=0.0015  diff(R-L)=+0.5857
  paired t-test:  t=132.3707, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=1396.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      1.9458

--- Per-class breakdown ---

=== Only ===

Only: car  (N=2183 objects)
  mean R=0.4951  mean L=0.2741  diff(R-L)=+0.2211
  paired t-test:  t=43.9641, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=149131.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      0.9410

Only: truck  (N=236 objects)
  mean R=0.1876  mean L=0.1701  diff(R-L)=+0.0175
  paired t-test:  t=1.0842, p=0.279374 (not significant)
  Wilcoxon:       W=5879.0000, p=0.485801 (not significant)
  Cohen's d:      0.0706

Only: van  (N=520 objects)
 

## How to read this

- **Only R vs Only L, p < 0.05 with mean diff negative (L > R)** -- L is
  more informative than R *on its own*. Supports "L is a useful single-
  channel summary," but a skeptic could argue L might just be capturing
  the same brightness/structure info that G or B already provide.
- **Mask R vs Mask L, p < 0.05 with Mask-L scores lower (R compensates
  better than L does)** -- this is the stronger claim. G and B are
  present in both cases; if removing L hurts more than removing R even
  with G/B still there, G/B cannot substitute for what L was providing.
  That is direct evidence L carries information distinct from G/B, not
  just "L happens to be informative."
- Report both. If only the 'Only' test is significant, temper the paper's
  language to "L is a strong single-channel proxy" rather than "L
  provides unique, non-redundant information."
